In [65]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [66]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "input.txt"
)

('input.txt', <http.client.HTTPMessage at 0x23b14b63d10>)

In [67]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [68]:
# demo hyperparameters

batch_size = 32
block_size = 256
max_iters = 5000
eval_interval = 250
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2

In [69]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [70]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s] # takes a string, outputs a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # takes a list of integers, outputs a string

In [71]:
data = torch.tensor(encode(text), dtype=torch.long)

In [72]:
n = int(len(data)*0.9)
train_data = data[:n]
val_data = data[n:]

In [73]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [74]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [75]:
B,T,C = 4,8,2
x = torch.randn((B,T,C))

Fundamentally, the language model tries to predict the next position in a sequence. If each position has information about the ones before it, the model has a clearer, more specific idea of what the next position could be. However, uniformly averaging all prior positions of a sequence to predict the next is not sufficient, as each position has the same implicit weight and therefore wouldn't provide a reasonable idea of how important they are. This means the model isn't told which prior positions matter more for predicting what comes next, which motivates learned content-dependent weighting instead of uniform averaging.

In [76]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, 0)

A matrix multiplication between a lower triangular matrix of the means per row (wei) and the initial weights (x) is a weighted combination of the t+1 rows of x, since every position after t+1 is zero. This allows a direct matrix multiplication with x over the entire row rather than looping through each position and finding the mean, which produces the same result as the nested loop but faster. However, this still doesn't solve the uniform-weighting problem from Section 1.

In [77]:
wei = torch.tril(torch.ones((T,T)))
wei /= wei.sum(1, keepdim=True)
xbow2 = wei @ x

Although the weights themselves are still uniform at this stage, a softmax function is applied so that when learned weights replace the uniform zeros, differences between them will be amplified. This is because of the exponentiation in softmax: it punishes the weaker weights and prioritizes the stronger, more important weights. Meanwhile, the -inf masking ensures that the rows after t+1 remain zero because every value is taken as an exponent, since e^(-inf) equals zero while e^0 equals one.

In [78]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x

This section illustrates single-head attention in NumPy with a query, key, and value. A query is what the model is trying to look for. A key is the metadata that determines the relevance of a position. A value is the actual content and information of what the query is looking for. Using two different matrices for a key and value allows for two independent degrees of freedom; if they were identical, then what the position says and what it actually contributes to the output would be the same no matter the context. Learned weights can be found by doing a matrix multiplication over the query and transposed key matrices, which is applied with a scaling factor head_size**-0.5 to ensure that softmax does not collapse to a one-hot distribution and vanish the gradients. This is then passed to softmax to amplify the differences and matrix multiplied again by the value to get the output.

In [79]:
B,T,C = 4,8,32
x = np.random.randn(B,T,C)
head_size = 16

In [80]:
class Linear:

    def __init__(self, fan_in, fan_out, bias=False):
        self.weight = np.random.randn(fan_in, fan_out)
        self.bias = np.zeros(fan_out) if bias else None

    def __call__(self, x): 
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

In [81]:
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x)
    logits = exp_x / np.sum(exp_x, axis=-1, keepdims=True)
    return logits

In [82]:
x = np.random.randn(B,T,C)
key_layer = Linear(C, head_size)
query_layer = Linear(C, head_size)
value_layer = Linear(C, head_size)
key = key_layer(x)
query = query_layer(x)
wei = query @ key.transpose(0, 2, 1) * head_size**-0.5

tril = np.tril(np.ones((T,T)))
mask = tril == 0

wei = np.where(mask, float('-inf'), wei)
wei = softmax(wei)
value = value_layer(x)
out = wei @ value

When we wrap the single-head attention in PyTorch from raw NumPy, we get a class with nn.Linear layers, which turns the data into tensors instead of arrays. While in NumPy there needed to be manual calculations of matrix multiplication and derivatives of the attentional operation, PyTorch's autograd allows for automatic gradient calculation since it tracks every operation on registered parameters.

In [52]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ('affinities')
        wei = q @ k.transpose(-2, -1) * C**-0.5 # (B,T,C) @ (B,C,T) --> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        wei = self.dropout(wei)
        # perform weighted aggregation of values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,T) @ (B,T,C) --> (B,T,C)
        return out

Multi-head attention, or running multiple attention heads in parallel,  is more efficient than simply increasing the size of a single head because it allows for multiple pieces of context to be processed at once. Take the sentence "the quick brown fox jumps over the lazy dog," for instance. In order to predict the last word "dog," different heads can specialize in different types of relationships at the same time: one head can process the word "fox" as the subject, another can process the word "jumps" as the verb, and so on. These heads can capture relationships such as syntactic roles, semantics, or positional proximity. However, a single head would have to compress all of that into one set of attention weights, losing information.

In [86]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

A block is the communication of the heads followed by computation, and they're stacked because the output of one gives the attention weights that each head already processed. This leaves the next block with context for hierarchical representation learning, rather than just giving the blocks raw input with no context. The feed-forward network computes a linear layer followed by a non-linearity on the data to model complex relationships. Since attention has already provided the context for the data, it is then passed to the feed-forward for the model to learn those relationships given the context. Residual connections add x to the final multi-head attention and feed-forward networks, which creates a direct route for gradients to flow from the final layer to the initial layer during backpropagation and preserves the loss. Layer norm is applied before multi-head attention and feed-forward to ensure consistent scale, preventing gradients from exploding or zeroing out during training.

In [87]:
# simple linear layer followed by non-linearity
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [88]:
# Transformer block: communication followed by computation
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        # n_embd: embedding dimension, n_head: number of heads we would like
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # x + is for residual connections
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [89]:
class Model(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits of the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_embd = self.token_embedding_table(idx) # (B,T,C) aka (Batch, Time, Channel)
        # pos_embd = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_embd #+ pos_embd # (B,T,C)
        x = self.blocks(x)
        logits = self.lm_head(x) # (B,T,vocab_size)
        if targets is None:
            loss = None

        else: 
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # crop idx to before block size
            idx_cond = idx[:, -block_size:]
            # get the predictions, idx is (B,T) array of indices in current context
            logits, loss = self(idx_cond)
            # last time step
            logits = logits[:, -1, :] # (B,C)
            # probabilities
            probs = F.softmax(logits, dim=-1) # (B,C)
            # new characters
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            # append sampled index to running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B,T+1)
        return idx

model = Model()
m = model.to(device)

In [57]:
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

In [58]:
for iter in range(max_iters):

    # every once in a while evluate loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f} val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.4485 val loss 4.4366
step 250: train loss 2.4173 val loss 2.4493
step 500: train loss 2.3470 val loss 2.3912
step 750: train loss 2.2978 val loss 2.3661
step 1000: train loss 2.1902 val loss 2.2723
step 1250: train loss 2.0678 val loss 2.1674
step 1500: train loss 1.9635 val loss 2.0690
step 1750: train loss 1.9120 val loss 2.0374
step 2000: train loss 1.8676 val loss 2.0162
step 2250: train loss 1.8315 val loss 1.9723
step 2500: train loss 1.7902 val loss 1.9506
step 2750: train loss 1.7914 val loss 1.9507
step 3000: train loss 1.7878 val loss 1.9432
step 3250: train loss 1.7678 val loss 1.9322
step 3500: train loss 1.7670 val loss 1.9190
step 3750: train loss 1.7676 val loss 1.9354
step 4000: train loss 1.7554 val loss 1.9195
step 4250: train loss 1.7525 val loss 1.9138
step 4500: train loss 1.7323 val loss 1.9081
step 4750: train loss 1.7430 val loss 1.9113


In [59]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=1000)[0].tolist()))


Al Thoum dishell chepaince and msortsen, thou stet,
Mowbling love braicks not pentenchs, if alstes,
And him no the miefrsek of in then of oyal. as!

These ne caubh, p tallave O, bequals that we Castitizes,
There feare wors upon tol melimy.
WARWIV:
Onceal, but sheel bupone, my sin I
Which wanould now hims; bear bnshearme.
HENRY Bunt.
TLINGBROKE:
We it is God no He!-have, Ye'tmocoust ysoufse.
FRARIAR ET:
Ay, nd my lof ard,
I windds. Hethat mat I rth whe shat orteratst, hour I hesinghe blan
You beng stind Befode lack? what no!
If, aowarnom to to sominow, but ofrom trongood cd me starle!
That tllet of thy to thath athiner gue all by she,
To stago ouraind hight f ca helapst he treruonr,
Upoph this in ead as-' thone, avend he kim incob,
I scaut n atewith fe he dheath t vo; to milet death we
asumo? tharaise dey eintis o tok-prn haveathery, y and provesus
'Seixtis tureat y byowhat ord;
A dankind urye t fon with incighamaben,
And his ing, will nour mon prosed kshaly.

AUFIDIUS:
Shose, gedid is

While learned positional embeddings are important for tracking the current position in a process, they can fail to capture the relative distance between positions and lose important information. RoPE (or Rotary Positional Embedding) rotates the embedding matrix so that the dot product between any two positions depends only on their relative distance, not their absolute positions. This can be verified by passing a constant vector through RoPE and confirming that dot products between positions with equal relative distance are identical.

In [83]:
class RoPE(nn.Module):

    def __init__(self, d, max_seq_len=128):
        super().__init__()
        theta = 10000 ** -(torch.arange(0,d,2).float() / d) # (1, d/2)
        m = torch.arange(max_seq_len) # (max_seq_len, 1)
        angles = torch.outer(m, theta).repeat_interleave(2, dim=-1) # (max_seq_len, d)
        self.register_buffer('angles', angles)

    def forward(self, q):
        seq_len = q.shape[1] # (seq_len)
        angles = self.angles[:seq_len] # (seq_len, d)
        q_swapped = torch.stack([-q[..., 1::2], q[..., ::2]], dim=-1).reshape_as(q)
        final_output = (q * torch.cos(angles)) + (q_swapped * torch.sin(angles))
        return final_output

In [84]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.rope = RoPE(head_size, max_seq_len=block_size)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.rope(self.key(x)) # (B,T,C)
        q = self.rope(self.query(x)) # (B,T,C)
        # compute attention scores ('affinities')
        wei = q @ k.transpose(-2, -1) * C**-0.5 # (B,T,C) @ (B,C,T) --> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        wei = self.dropout(wei)
        # perform weighted aggregation of values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,T) @ (B,T,C) --> (B,T,C)
        return out

In [85]:
rope = RoPE(d=16, max_seq_len=128)
q = torch.ones(1, 10, 16)
q_rot = rope(q)
print((q_rot[0, 2] @ q_rot[0, 5]).item())  # distance 3
print((q_rot[0, 0] @ q_rot[0, 3]).item())  # distance 3
print((q_rot[0, 1] @ q_rot[0, 6]).item())  # distance 5
print((q_rot[0, 0] @ q_rot[0, 5]).item())  # distance 5

11.086202621459961
11.086202621459961
12.274080276489258
12.274080276489258


In [90]:
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

In [91]:
for iter in range(max_iters):

    # every once in a while evluate loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f} val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.4132 val loss 4.4126
step 250: train loss 1.8965 val loss 1.9760
step 500: train loss 1.6680 val loss 1.8117
step 750: train loss 1.5708 val loss 1.7494
step 1000: train loss 1.5078 val loss 1.6938
step 1250: train loss 1.4697 val loss 1.6726
step 1500: train loss 1.4381 val loss 1.6327
step 1750: train loss 1.4134 val loss 1.6160
step 2000: train loss 1.3893 val loss 1.5921
step 2250: train loss 1.3730 val loss 1.5833
step 2500: train loss 1.3615 val loss 1.5698
step 2750: train loss 1.3510 val loss 1.5678
step 3000: train loss 1.3373 val loss 1.5518
step 3250: train loss 1.3304 val loss 1.5425
step 3500: train loss 1.3196 val loss 1.5537
step 3750: train loss 1.3098 val loss 1.5390
step 4000: train loss 1.3046 val loss 1.5380
step 4250: train loss 1.2960 val loss 1.5304
step 4500: train loss 1.2937 val loss 1.5459
step 4750: train loss 1.2892 val loss 1.5232


In [92]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=1000)[0].tolist()))


Kind, thus as teal the heaven them should not
Shall hid we can, get us person and piece
Say.

BARYIUS:
A-my hardler, which dover'd in Ludio, I chank you
Too pircharded muster.

WARCIUS:
Last
Fortuth, sir, early, a boat Pers,
As if itself: it course is younds as England,
In stings man she in hot, these given a war,
Who says deeds, that yet well to-morrow.

LEONTES:
Good didspairer: I privage, sir
Wanting shall: or yet with, sistings are.

POMPEY:
Sir! I out heard for him, for I would loved him nothing
Ere incline thine with winters, the lords yet good;
In his, by us in thy chance loves a man;
But, so, Claudior be spirit, undeed
we salupe of this worthing is, and restandsike,
Which you you progation to night that eye,
Unlike remelments poor treasons,
They letter shall be says shall revenge
Be sleeps and mine facend man'd;
And will shear Corioli: when who hath timeth
Enlow he can the beggaring of Saur mayster,
Lattence for the grove, master not san,
though I thank which you have my queen

Demo comparison (n_embd=128, n_head=4, n_layer=4, block_size=256, max_iters=5000)

Base GPT:  train loss 1.7430 | val loss 1.9113

RoPE GPT:  train loss 1.2892 | val loss 1.5232

The ablation experiments removed different features of the model (in this case, positional embeddings, residual connections, and scaling factor). I ran them to explore how the model may or may not break down with the lack of these features. By removing one component at a time while keeping everything else constant, we can isolate the contribution of each component independently. The following notes show the three experiments, their results, and an explanation for each finding.

regular - train loss 2.2788 val loss 2.2906


no positional embedding - train loss 2.2764 val loss 2.2869

 - decoder-only transformers can still learn surprisingly well without PE

 - model is so small it memorizes local patterns and can pick up cues from token statistics


no residual connections - train loss 3.1892 val loss 3.1796

 - gradients had to pass through numerous nonlinear layers, leading to destruction of features and lack of representative continuity instead of refining representations
   
 - each layer overwrites representations rather than refining them, led to more unstable optimization


no scaling factor - train loss 2.2640 val loss 2.2766

 - since head_size (C) = 4, -sqrt(C) = 1/2, so we only divide by two
   
 - operation is too tiny to have any meaningful effect

Best result — full scale training

Hyperparameters: n_embd=384, n_head=6, n_layer=6, block_size=256, max_iters=5000

Train loss: 0.9633 | Val loss: 1.5242

hat died must cannot be born. Thou speak again,
As is the pledge in an assay to take
Which thou hast finded divine those my wretched?
So, for wilt Thomas hither and come to cheer
The queen's ears doth fall a vial grave
For Ravenspurghbor, and my raged grudges,
Known life is and company to that the
Watchmon poor Six and ducked on us,
I had you not pass again of nothing
Shall visit the my country, and rob ounter.

CAMILLO:
That is more at as they have done power them; I
Virtak not the birth. My villain, I loved thee
yet thither: and, and thou caut married,
Let them no scape of my sinjure bends now.

ROMEO:
Pray look, I thank thee soft myself;
For his father's face, he will never incannot.

QUEEN MARGARET:
O, be content: O me this lord, thou art dispotiate.

KING RICHARD II:
Then most I strip the sweeter's lord,
Shout you not.

KING RICHARD III:
Tears are bemised my lords with villain;
Which they shall like you attending him,
Shall since to Ledd.

First Huntsman:
The way will not call the heaven lay 'nothing.'

KING HENRY VI:
His noble lord can sweet York, I hope:
In Edwardlis! not remover him; we'll have king.

BUCKINGHA:
Hadst thou be gone, sir, that sense his husband,
Look in Saint John Partley, Duke of Norfolk:
He twist flatterers, when you confour there;
Men, as he spaked as for your sweet seen:
From London and first of Hereford,
I the pender wilful and have to hear,
His present made in Verona old:
I praite thee, boy; I that here I leate talk
Raged thee, are in a sin honour in my cheeks:
Maste way at pains in that I dangerous;
Against thou Henry super a victory.

KING RICHARD III:
Your might died,
Is not yet nature, had such enreation.

NORTHUMBERLAND:
What would you up; and your good Tower?

CLARENCE:
No; for the respectain war lions there;
Six does change congeners than Edward's silence.

KING RICHARD II:
Nay, by, uncle; and that Warwick's conscience? O,
Unbeing and thy fown: light is for the fools,
And force 'twas courteous streets in the coron,
Wheres our curenners stoe, welcome, zes,
Or wide the world sesting of his limit,
And never severe's haught, nor to do this death
To Itare begin that moveth at all.

First Lord:
Where is the lempteth peace would take him there;
Fsely, he told any hard: ame this young disclaims,
It drows not moved, methinks, subdued to see.
Will come take mine; I am am Gaunt in king.
Let me enemy and enforce and ingrate,
Tullest thou yield us to the people forget.
What then before the mercy that where it me?
Dare he not do't. Nay, boy? you what care
To wail himstand by and larmoning?

Garron:
Why, my soother?

JULIET:
More than doth clouds; and look's in growing,
And for the slew of sweet with hours up,
If thou discontented, chide to the fire,
Where you best inclined, none by young again.
Fates to the wars; what news?

RIVEL:
Misery;
And should she sweet use in it, sir.
Must die to-morrow; with pitch'd the father
Had stand upon the other; let's by us oint.
His eyes in the lusty down earth; his own deeds
And in the approachor.

DUKE VINCENTIO:
I vouge you, then?

LUCIO:
My lord, on honour of a tainous bitter.

ISABELLA:
Away, we will put you with a breathed:
A pont you are aside, and hurt mee before:
'Tear as is thy lovely in post, by meet,
And prite but safely on thy cheek thee will?
Hath that contract this my other again,
And am I dispatch'd more fruit-trees, wretched
And in the enemies of my true clear'.

KING EDWARD IV:
Ay, and Ragard Clifford, in vail of England,
I' tread to Bolingbroke, when he shall arm me.

KING RICHARD:
I prack my own, despairing all accustors.

GLOUCESTER:

KING EDWARD IV:
Brother, farewell, and the pen is and emit:
Yea, revengeance the king my son of York.

QUEEN MARGARET:
O more! dare not entreaty to a fither,
As Isabel, are the people that dost thou negabor,
The sorrow in thy tumb and ignoble quiet:
For that thou name to know his child and Willian:
For tell there, thy worth repairs not to thine;
For here we would go it; one hollow not thine.
Will I not be heard? Edward thee not?
Alas, you thither none and revenged,
And after the rest are, if they know thou hast
The thigone to gainst so lintelless
And made her sturved was a bloody wedged,
Shall not rise and do well met; fove and fled,
Was knock, or oath wars cannot noble and worth.

QUEEN MARGARET:
Stay brother Forberties, Jo: who conversation
At those my son shall be poison tells of the world,
By father'd light. But for the regal throne,
That lie a town death on the anger grow?

EDERSE:
Where is their cousin reconcile arms lost,
That Edward willight upon the safewy throne.
The deadly points of foes! altorments
Breake on Lamentation of the fiend swear.

QUEEN MARGARET:
And, ay, my cousin looks I come to thee.

KATHARINA:
Three-protector, I will she four about?

KING RICHARD II:
In water, Warwick: and still, my uncle,
So true to ErV:
How after thou canst Lord Hastings, is provort:
Will't you ghost a change more, my lord,
To come the French with a hope of Iriaw,
And bid what tender penation
End Corioli: England, I'll resign for yourselves,
'Tis gruer more than fellow.' I came into you:

Clown:
To think a port at expection. And yet thou been
Where you reeads again borth, most I receive
Than so dishonoured: say, 'twill take the life.
Uncle me with one shower from Thur, as do more.
While we mean Edward's hope.

PRINCE:
You put but his either

Nurse:
Well, we will welcome I see yours.

Ourse:
Yes it is: despress aloned umonar?

LUCIO:
'Tybalt,' good seas! thy remedy, Juday! Love me speech.

ISABELLA:
That's a womb; since thy word, unstilled.

DUKE OF AUMERLE:
I must prove see my thanks. When men, do you cause
Be now my enter in the trial, that I mean thee.
Flack on fasts my humble is this; and, she's blessed
She to the way like writing, or who shall be nail.

DUKE OF YORK:
It beseeming in their abominable.

DUKE OF YORK:
To save this life
Great king, and I am no more accused:
Villain help in two touching by ears,
When I have done my own perfect to see
And traial triumphs heaven and true graces
Destroved himself, slaughter, her father's son.
O, now have VIUS:
I would not have deserved: if I prevail' and my best
me and senses at this spirt. Come, how can ye
parth to here, patience were not in their hands.

AUTOLYCUS:
O, the young after gisle me not that fair incline.

Citizens:
To take now she is idle, the ear; strike sins,
Do can paid Coriolanus.

Second Citizen:
By tale thine own good father.

Citizens:
Care you are.

Citius:
To-morrow, sir, sir, as the king, had a pardon'd.

Third Officer:
So thrive awful as you should do the gods.

Second Citizen:
There larks you know her enemies, and the partner
hollong yourselves, we will stole on purge other.

Third Citizen:
Why, Hark you me! The men in Elizabe sets of heaven,
they shall be roaded with one depart mades; and
When ever Mawgaret, all exercise Paul,
I know not, and to the gove-bed for the grave.

HENRY BOLINGBROKE:
I know thy words.

KING HENRY VI:
Farewell, my mother your company straight,
Where fith the seems hath by maed but us the
Furwardned so of thy birthright arguments
And by this differences to thy destroy.
First, thou never years can we be possess?
Why hold, or dignite-tree, condemned after!
Thy loving friends spirit with age veing Herest.

FLORIZEL:
My lord, the time I can:
But it makes for with me pals after tyrants,
And never feed I have heard, and lend now
She is confidence again.

CLAUDIO:
This shast piteous house,
He prevents and pluck himself a way.

LUCIO:
The truth is noble
I'll resign in the apel.

ISABELLA:
Breauteous!
Come, cold, strike, awake to take about thy death:
Thou art proonet suggarsely, and hope their daughter.
The quiet full of such patbleck to hide,
Uncle finds for centre alrege,
Believe him speaks. To
Whou were in Tranio; intentious of those sentence,
in the people, in anger, and bear the dead,
Yet seek can speak to 'Though than deserve:
His welcomes mischance and England, yet true
For this title joint to another ede;
I know they have to't for thy head i' the hand
To crook enbrain; be bailisting deterimed,
Which thle in advance hot, I may;
Had not too allowed with an most profort!

Second Watchman:
So erey, to the oTow?

BUCKINGHAM:
When do after, was he was delibed?

KING HENRY VI:
Nay, Bagot.

KING RICHARD II:
Cut of Hermione that hath you so?

BUCKINGHAM:
Hois earth and in the court,
Which he not for me to-negs revenge beauty.

KING RICHARD III:
Warwick, but that sudden conceror,
I'll then away the place of them and women
To hear or never came to thee, made thee of.

KING LEWIS XI:
For Warwick, the bosom of your kindre,
I give them a duke. Tell me your grace fear: for them
He was Hereford me fit with him you have wrong'd
Limbs one execute; for then was I took above
And you and hold in yours.

POMUCESTER:
I will not Pomfret,
To the good bemies of Angelo; amen!

LADY ANNE:
To look you, methought,
Is hateful sacred; never will to speak to thee
Thus have recumposed not an employ!

SALY:
Here comes about the base doth not pake him a
pronement his father to give his fearful books.
Ah, hark in these transport Harpine me,
That no, I have done a thing spoken as so
But negling with an crowish his deeds.

First Murderer:
Back his no more than neg's desiring gild.

First Citizen:
And what success that is for both Lord Bolina?

MERCUTIO:
Here is con by to-morrow: if this banish'd
Will may I thank thee, not only Bianca,
Your dare contented son to weep the dead.

HENRY BOLINGBROKE:
I chang his with to--More with my shadow he would devise
Then needs that cannot be delighted.
Write on me from the fiery worthy land,
When thou might were not--parted food up do the reed
I then thee be a wise-man that I speak with thee.

KING RICHARD III:
And leave me, and bid us, go hath;
Thy virtuous manmented shall dead by gone.

DUKE VINCENTIO:
But, I did not such advanced their events and true months,
Die foregood princely be no more than let's,
Thou art assure me we do conside, that seests
Wels have the young as ares! If this staff,
Have more than anothe